# Cynbit DocuMind AI\n## Notebook 01: Document Ingestion\n\n### Objective\nThis notebook builds the first stage of DocuMind: converting a PDF into page-level, machine-readable records. Each record retains text, source filename, page number, character count, document metadata, and an empty-page flag.\n\n### Why this matters\nRetrieval and answer generation need traceable source text. Keeping page boundaries now lets later RAG answers cite the page that supported them.

### 1. Imports\nPyMuPDF reads PDF pages; `Path` keeps paths portable; pandas provides a compact inspection table.\n\n**Expected output:** `All libraries imported successfully!`

In [ ]:
import sys\nfrom pathlib import Path\n\nimport pandas as pd\n\nPROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nfrom app.ingestion.pdf_loader import extract_pdf_text\nprint('All libraries imported successfully!')

### 2. Configure project paths\nUsing paths relative to the project root avoids hard-coding a specific Mac username or folder.

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data'\nSAMPLE_DOCS_DIR = DATA_DIR / 'sample_documents'\nSAMPLE_DOCS_DIR.mkdir(parents=True, exist_ok=True)\nprint('Project Root:', PROJECT_ROOT)\nprint('Sample Documents:', SAMPLE_DOCS_DIR)

### 3. Extract and inspect a PDF\n`extract_pdf_text` returns a list rather than one large string, so every text segment stays linked to its originating page. Blank or scanned pages are safely retained with `has_text=False`; OCR can be added later.\n\n**Validation:** the summary should show one row per PDF page, and page numbers should start at 1.

In [ ]:
def inspect_extraction(extracted_pages):\n    return pd.DataFrame([\n        {key: page[key] for key in ('page_number', 'character_count', 'has_text', 'source_file')}\n        for page in extracted_pages\n    ])\n\npdf_files = sorted(SAMPLE_DOCS_DIR.glob('*.pdf'))\nprint('PDF documents found:', len(pdf_files))\nif pdf_files:\n    extracted_pages = extract_pdf_text(pdf_files[0])\n    display(inspect_extraction(extracted_pages))\n    print('\\nFirst page preview:\n')\n    print(extracted_pages[0]['text'][:1000])\nelse:\n    print('Add a PDF to data/sample_documents and run this cell again.')

### 4. Automated validation\nRun the following from the project root to validate normal extraction, blank-page handling, missing files, and non-PDF rejection: `venv/bin/python -m pytest -q`.